# PageRank Graph Python Implementation

In [6]:
import numpy as np
import networkx as nx

In [7]:
class Graph:
    def __init__(self):
        self.nodes= []

    def contains(self, name):
        for node in self.nodes:
            if(node.name== name):
                return True
        return False

    # Return the node with the name, or create and return a new node if not found
    def find(self, name):
        if(not self.contains(name)):
            new_node= Node(name)
            self.nodes.append(new_node)
            return new_node
        else:
            return next(node for node in self.nodes if node.name== name)

    def add_edge(self, parent, child, weight):
        parent_node= self.find(parent)
        child_node= self.find(child)

        parent_node.link_child(child_node)
        child_node.link_parent(parent_node, weight)

    def print_graph(self):
        node= self.nodes
        
        for i in range(len(self.nodes)):
            print(node[i].name, end= '')
            parent= node[i].parents
            weight= node[i].edge_ws
            
            for j in range(len(node[i].parents)):
                if (j> 0):
                    print("    is linded to", parent[j].name, "edge weight", weight[j])
                else:
                    print(" is linded to", parent[j].name, "edge weight", weight[j])
                    
    def display(self):
        node= self.nodes
        G= nx.DiGraph()
        data= []
        
        for i in range(len(self.nodes)):
            child= node[i].name
            parents= node[i].parents
            weights= node[i].edge_ws
            
            for j in range(len(node[i].parents)):
                parent= parents[j].name
                weight= weights[j]
                
                data.append([parent, child, weight])
            
        G.add_weighted_edges_from(data)
        pos= nx.spring_layout(G)
        nx.draw(G, pos, with_labels=True, node_size=500, edge_color='#eb4034', width=2, font_size=12, font_weight=50, arrowsize=10, alpha=0.8)

    def sort_nodes(self):
        self.nodes.sort(key=lambda node: int(node.name))

    def display_hub_auth(self):
        for node in self.nodes:
            print(f'{node.name}  Auth: {node.old_auth}  Hub: {node.old_hub}')

    def normalize_auth_hub(self):
        auth_sum= sum(node.auth for node in self.nodes)
        hub_sum= sum(node.hub for node in self.nodes)

        for node in self.nodes:
            node.auth /= auth_sum
            node.hub /= hub_sum

    def normalize_pagerank(self):
        pagerank_sum= sum(node.pagerank for node in self.nodes)

        for node in self.nodes:
            node.pagerank /= pagerank_sum

    def get_auth_hub_list(self):
        auth_list= np.asarray([node.auth for node in self.nodes], dtype='float32')
        hub_list= np.asarray([node.hub for node in self.nodes], dtype='float32')

        return auth_list, hub_list

    def get_pagerank_list(self):
        pagerank_list= np.asarray([node.pagerank for node in self.nodes], dtype='float32')
        return pagerank_list

In [3]:
class Node:
    def __init__(self, name):
        self.name= name
        self.children= []
        self.parents= []
        self.edge_ws= []
        self.auth= 1
        self.hub= 1
        self.pagerank= 1

    def link_child(self, new_child):
        for child in self.children:
            if(child.name == new_child.name):
                return None
        
        self.children.append(new_child)

    def link_parent(self, new_parent, weight):
        for parent in self.parents:
            if(parent.name == new_parent.name):
                return None
        
        self.parents.append(new_parent)
        self.edge_ws.append(weight)

    def update_auth(self):
        self.auth= sum(node.hub for node in self.parents)

    def update_hub(self):
        self.hub= sum(node.auth for node in self.children)

    def update_pagerank(self, d, n):
        in_neighbors= self.parents
        w_in_neighbs= self.edge_ws
        pagerank_sum= 0
        
        for i in range(len(in_neighbors)):
            pJ= in_neighbors[i].pagerank
            wJ= w_in_neighbs[i]
            lJ= len(in_neighbors[i].children)
            
            pagerank_sum += wJ* (pJ/ lJ)
            
        random_jump= (1-d)/ n
        self.pagerank= random_jump+ (d)* pagerank_sum

In [4]:
def init_graph(filename):
    with open(filename) as f:
        lines= f.readlines()

    graph= Graph()

    for line in lines:
        [parent, child, weight]= map(float, line.strip().split(','))
        
        graph.add_edge(parent, child, weight)

    graph.sort_nodes()

    return graph